# Taco Hemingway Album Classifier

In this notebook I build a simple NLP classifier that predicts the album of a Taco Hemingway song from its lyrics.

I use TF-IDF features, classic scikit-learn models, cross-validation, model comparison, error analysis and feature inspection. The goal is not only to maximize accuracy, but to show a complete and honest ML workflow on a small, imbalanced text dataset.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
TEST_SIZE = 0.2
MIN_SONGS_PER_ALBUM = 4

data_candidates = list(Path("/kaggle/input").rglob("lyrics_data.csv"))
if not data_candidates:
    raise FileNotFoundError("lyrics_data.csv not found under /kaggle/input")

DATA_PATH = data_candidates[0]
print(DATA_PATH)

## Load and inspect data

In [ ]:
df = pd.read_csv(DATA_PATH, sep=";;", engine="python")
df = df.dropna(subset=["album", "lyrics"]).copy()
df["album"] = df["album"].astype(str).str.strip()
df["title"] = df["title"].astype(str).str.strip()
df["lyrics"] = df["lyrics"].astype(str).str.strip()
df = df[df["lyrics"].str.len() > 0].copy()

print(df.shape)
display(df[["album", "title"]].head())

album_counts = df["album"].value_counts()
display(album_counts.to_frame("songs"))

ax = album_counts.sort_values().plot(kind="barh", figsize=(10, 7), title="Songs per album")
ax.set_xlabel("Number of songs")
plt.tight_layout()
plt.show()

## Filter small classes

I keep albums with at least 4 songs. This preserves 15 albums and removes only records that would make stratified evaluation unstable.

In [ ]:
valid_albums = album_counts[album_counts >= MIN_SONGS_PER_ALBUM].index
filtered = df[df["album"].isin(valid_albums)].copy()

X = filtered["lyrics"].astype(str)
y = filtered["album"].astype(str)

print("Albums:", y.nunique())
print("Songs:", len(filtered))

## Train/test split and baseline

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

min_class_count = y.value_counts().min()
cv = StratifiedKFold(
    n_splits=min(5, min_class_count),
    shuffle=True,
    random_state=RANDOM_STATE,
)

def build_pipeline(classifier, ngram_range=(1, 2), min_df=2, max_features=5000, sublinear_tf=True):
    return Pipeline(
        steps=[
            (
                "tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    max_df=0.95,
                    min_df=min_df,
                    max_features=max_features,
                    ngram_range=ngram_range,
                    sublinear_tf=sublinear_tf,
                ),
            ),
            ("classifier", classifier),
        ]
    )

## Model comparison

In [ ]:
models = {
    "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
    "logistic_regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="lbfgs",
        random_state=RANDOM_STATE,
    ),
    "multinomial_nb": MultinomialNB(),
    "linear_svc": LinearSVC(class_weight="balanced", random_state=RANDOM_STATE),
}

rows = []
for model_name, classifier in models.items():
    pipeline = build_pipeline(classifier)
    cv_scores = cross_val_score(pipeline, X, y, cv=cv, scoring="accuracy", n_jobs=1)
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    rows.append(
        {
            "model": model_name,
            "cv_accuracy_mean": cv_scores.mean(),
            "cv_accuracy_std": cv_scores.std(),
            "test_accuracy": accuracy_score(y_test, predictions),
            "test_macro_f1": f1_score(y_test, predictions, average="macro", zero_division=0),
            "test_weighted_f1": f1_score(y_test, predictions, average="weighted", zero_division=0),
        }
    )

comparison = pd.DataFrame(rows).sort_values("test_macro_f1", ascending=False)
display(comparison)

## Main model evaluation

In [ ]:
main_model = build_pipeline(
    LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="lbfgs",
        random_state=RANDOM_STATE,
    )
)
main_model.fit(X_train, y_train)
predictions = main_model.predict(X_test)

print(classification_report(y_test, predictions, zero_division=0))

labels = sorted(y.unique())
matrix = confusion_matrix(y_test, predictions, labels=labels)
fig, ax = plt.subplots(figsize=(14, 14))
display_matrix = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=labels)
display_matrix.plot(ax=ax, xticks_rotation=90, values_format="d", colorbar=False)
plt.title("Confusion matrix - Taco Hemingway album classifier")
plt.tight_layout()
plt.show()

## Error analysis

In [ ]:
errors = pd.DataFrame(
    {
        "title": filtered.loc[y_test.index, "title"],
        "true_album": y_test,
        "predicted_album": predictions,
    }
)
errors = errors[errors["true_album"] != errors["predicted_album"]].copy()

print("Wrong predictions:", len(errors))
display(errors.head(15))
display(
    errors.groupby(["true_album", "predicted_album"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

## Feature inspection

For the linear model I can inspect the highest-weighted TF-IDF features per album. This is not causal explanation, but it helps check whether the model uses sensible textual signals.

In [ ]:
vectorizer = main_model.named_steps["tfidf"]
classifier = main_model.named_steps["classifier"]
feature_names = vectorizer.get_feature_names_out()

top_rows = []
for class_index, album in enumerate(classifier.classes_):
    weights = classifier.coef_[class_index]
    top_indices = weights.argsort()[::-1][:10]
    top_rows.append(
        {
            "album": album,
            "top_features": ", ".join(feature_names[index] for index in top_indices),
        }
    )

top_features = pd.DataFrame(top_rows)
display(top_features)

## TF-IDF experiments

In [ ]:
experiments = [
    ("default", (1, 2), 2, 5000, True),
    ("unigrams_only", (1, 1), 2, 5000, True),
    ("min_df_1", (1, 2), 1, 5000, True),
    ("max_features_3000", (1, 2), 2, 3000, True),
    ("no_sublinear_tf", (1, 2), 2, 5000, False),
]

experiment_rows = []
for name, ngram_range, min_df, max_features, sublinear_tf in experiments:
    pipeline = build_pipeline(
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            solver="lbfgs",
            random_state=RANDOM_STATE,
        ),
        ngram_range=ngram_range,
        min_df=min_df,
        max_features=max_features,
        sublinear_tf=sublinear_tf,
    )
    cv_scores = cross_val_score(pipeline, X, y, cv=cv, scoring="accuracy", n_jobs=1)
    pipeline.fit(X_train, y_train)
    exp_predictions = pipeline.predict(X_test)
    experiment_rows.append(
        {
            "experiment": name,
            "ngram_range": str(ngram_range),
            "min_df": min_df,
            "max_features": max_features,
            "sublinear_tf": sublinear_tf,
            "cv_accuracy_mean": cv_scores.mean(),
            "test_accuracy": accuracy_score(y_test, exp_predictions),
            "test_macro_f1": f1_score(y_test, exp_predictions, average="macro", zero_division=0),
        }
    )

tfidf_results = pd.DataFrame(experiment_rows).sort_values("test_macro_f1", ascending=False)
display(tfidf_results)

## Conclusions

The Logistic Regression model clearly beats the most-frequent baseline and remains a strong reference model for this small text classification task.

The task is difficult because the dataset is small, imbalanced and contains 15 classes. For that reason I treat macro F1, cross-validation and error analysis as more important than raw accuracy alone.

In this split, the unigram-only TF-IDF setup performs better than the default unigram+bigram setup, so it is a good candidate for a follow-up experiment.